# # 06. Full DMS Pipeline — Abeta42 + IAPP + CANYA
# 모든 실험을 하나로. 셀 단위로 코랩에 복붙.



In [ ]:
# CELL 1: Setup
import os, sys
os.chdir('/content')
!rm -rf brain_idp_flow
!git clone https://github.com/layeredlabs06/brain-idp-flow.git brain_idp_flow
os.chdir('brain_idp_flow')
!pip install -e . -q
!pip install fair-esm scipy scikit-learn openpyxl -q
sys.path.insert(0, '/content/brain_idp_flow/src')

import torch, numpy as np, yaml, glob, os
from scipy.stats import spearmanr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 2: Drive 마운트 + 체크포인트 확인
from google.colab import drive
drive.mount('/content/drive')

CKPT_35M = '/content/drive/MyDrive/brain_idp_flow_ckpts/ckpt_best.pt'
CKPT_650M = '/content/drive/MyDrive/brain_idp_flow_ckpts_650m/ckpt_best.pt'

print(f'35M ckpt exists: {os.path.exists(CKPT_35M)}')
print(f'650M ckpt exists: {os.path.exists(CKPT_650M)}')

In [ ]:
# CELL 3: 모델 로드 (35M + 650M)
from brain_idp_flow.sample import load_model, sample_ensemble_with_trajectory
from brain_idp_flow.features.seq_embed import ESM2Embedder
from brain_idp_flow.geometry.metrics import radius_of_gyration
from brain_idp_flow.data.dataset import ProteinEnsembleDataset
from brain_idp_flow.analysis.trajectory_analysis import extract_trajectory_features

with open('configs/flow.yaml') as f:
    config = yaml.safe_load(f)
with open('configs/flow_650m.yaml') as f:
    config_650m = yaml.safe_load(f)

model = load_model(config, CKPT_35M, device)
embedder = ESM2Embedder(device=device)
print('35M model loaded')

model_650m = load_model(config_650m, CKPT_650M, device)
embedder_650m = ESM2Embedder(model_name='esm2_t33_650M_UR50D', device=device)
print('650M model loaded')

In [ ]:
# CELL 4: Abeta42 DMS 데이터 로드
from brain_idp_flow.data.dms_loader import load_seuma_dms

os.makedirs('data/dms', exist_ok=True)
dms_data = load_seuma_dms(cache_dir='data/dms')
print(f'Abeta42 DMS: {len(dms_data)} mutations')
print(f'fAD mutations: {sum(1 for d in dms_data if d["is_fad"])}')

ABETA_SEQ = 'DAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVVIA'

In [ ]:
# CELL 5: Abeta42 35M Scoring (오래 걸림 — ~30-40분)
wt_emb = embedder.embed_single(ABETA_SEQ)
wt_result = sample_ensemble_with_trajectory(
    model, wt_emb, mut_pos=0, mut_aa=0,
    n_samples=64, n_trajectory_samples=16, device=device,
)
wt_rg = radius_of_gyration(torch.from_numpy(wt_result['ensemble'])).mean().item()
print(f'WT Rg (35M): {wt_rg:.2f}')

scored_dms = []
for i, mut in enumerate(dms_data):
    mut_seq = list(ABETA_SEQ)
    mut_seq[mut['pos'] - 1] = mut['mt']
    mut_seq = ''.join(mut_seq)
    mut_emb = embedder.embed_single(mut_seq)
    aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut['mt'], 0)

    result = sample_ensemble_with_trajectory(
        model, mut_emb, mut_pos=mut['pos'], mut_aa=aa_idx,
        n_samples=32, n_trajectory_samples=4, n_steps=50, device=device,
    )
    mut_rg = radius_of_gyration(torch.from_numpy(result['ensemble'])).mean().item()
    traj_feats = extract_trajectory_features(result['trajectory'], mutation_pos=mut['pos'] - 1)

    scored_dms.append({
        **mut,
        'delta_rg': mut_rg - wt_rg,
        **{k: v for k, v in traj_feats.items() if not k.startswith('_')},
    })
    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{len(dms_data)}')

print(f'35M Done: {len(scored_dms)}')

In [ ]:
# CELL 6: ESM-2 LLR
from brain_idp_flow.analysis.esm2_llr import ESM2MutationScorer

scorer = ESM2MutationScorer(device=device)
for i, mut in enumerate(scored_dms):
    llr_result = scorer.score_mutation(ABETA_SEQ, mut['pos'], mut['wt'], mut['mt'], fast=True)
    mut['llr_site'] = llr_result['llr_site']
    if (i + 1) % 200 == 0:
        print(f'  LLR {i+1}/{len(scored_dms)}')
print('LLR done')

In [ ]:
# CELL 7: Abeta42 650M Scoring
wt_emb_650 = embedder_650m.embed_single(ABETA_SEQ)
wt_result_650 = sample_ensemble_with_trajectory(
    model_650m, wt_emb_650, mut_pos=0, mut_aa=0,
    n_samples=64, n_trajectory_samples=4, device=device,
)
wt_rg_650 = radius_of_gyration(torch.from_numpy(wt_result_650['ensemble'])).mean().item()
print(f'WT Rg (650M): {wt_rg_650:.2f}')

scored_650m = []
for i, mut in enumerate(dms_data):
    mut_seq = list(ABETA_SEQ)
    mut_seq[mut['pos'] - 1] = mut['mt']
    mut_seq = ''.join(mut_seq)
    mut_emb = embedder_650m.embed_single(mut_seq)
    aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut['mt'], 0)

    result = sample_ensemble_with_trajectory(
        model_650m, mut_emb, mut_pos=mut['pos'], mut_aa=aa_idx,
        n_samples=32, n_trajectory_samples=4, n_steps=50, device=device,
    )
    mut_rg = radius_of_gyration(torch.from_numpy(result['ensemble'])).mean().item()
    scored_650m.append({**mut, 'delta_rg_650m': mut_rg - wt_rg_650})
    if (i + 1) % 100 == 0:
        print(f'  650M: {i+1}/{len(dms_data)}')

print(f'650M Done: {len(scored_650m)}')

In [ ]:
# CELL 8: Abeta42 Correlation Table
import matplotlib.pyplot as plt

ns = np.array([d['nucleation_score'] for d in scored_dms])

features_to_test = [
    ('llr_site', 'ESM-2 LLR'),
    ('delta_rg', 'Delta Rg (35M)'),
    ('late_velocity_site', 'Late-Stage Velocity'),
    ('velocity_variance_late', 'Velocity Variance'),
    ('switching_rate_site', 'Contact Switching'),
    ('switching_rate_long_range', 'Long-Range Switching'),
    ('convergence_time_site', 'Convergence Time'),
]

print('=' * 60)
print(f'ABETA42 DMS CORRELATION (n={len(scored_dms)})')
print('=' * 60)
for feat_key, feat_name in features_to_test:
    vals = np.array([d.get(feat_key, 0) for d in scored_dms])
    if vals.std() < 1e-10:
        continue
    rho, pval = spearmanr(vals, ns)
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f'  {feat_name:<30} rho={rho:>7.3f}  p={pval:.2e} {sig}')

# 650M
drg_650m = np.array([d['delta_rg_650m'] for d in scored_650m])
rho_650, p_650 = spearmanr(drg_650m, ns)
print(f'  {"Delta Rg (650M)":<30} rho={rho_650:>7.3f}  p={p_650:.2e} ***')
print('=' * 60)

In [ ]:
# CELL 9: 35M vs 650M 비교 플롯
drg_35m = np.array([d['delta_rg'] for d in scored_dms])
rho_35m, p_35m = spearmanr(drg_35m, ns)
is_fad = np.array([d.get('is_fad', False) for d in scored_dms])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, drg, rho, p, title in [
    (axes[0], drg_35m, rho_35m, p_35m, 'ESM-2 35M Flow Model'),
    (axes[1], drg_650m, rho_650, p_650, 'ESM-2 650M Flow Model'),
]:
    ax.scatter(drg[~is_fad], ns[~is_fad], s=10, alpha=0.4, c='steelblue')
    if is_fad.any():
        ax.scatter(drg[is_fad], ns[is_fad], s=80, facecolors='none',
                   edgecolors='red', linewidth=2, label='fAD')
    ax.set_xlabel('delta Rg'); ax.set_ylabel('Nucleation Score')
    ax.set_title(f'{title}\nrho={rho:.3f}, p={p:.2e}'); ax.legend(fontsize=9)
fig.suptitle('35M vs 650M: delta Rg vs Nucleation Score', fontsize=14)
fig.tight_layout()
os.makedirs('results/dms', exist_ok=True)
fig.savefig('results/dms/35m_vs_650m.png', dpi=150); plt.show()

In [ ]:
# CELL 10: Lean Composite + ML
from brain_idp_flow.analysis.ml_predictor import run_lean_composite, run_full_ml_pipeline

os.makedirs('results/ml', exist_ok=True)
lean_results = run_lean_composite(scored_dms, output_dir='results/ml')
print()
dms_ml_results = run_full_ml_pipeline(scored_dms, output_dir='results/ml')

In [ ]:
# CELL 11: Cross-Protein Transfer (tau, alpha-syn)
targets = load_targets('configs/targets.yaml')
disease_data = []

for tid, target in targets.items():
    if tid == 'abeta42':
        continue
    print(f'\nScoring {target.name}...')
    wt_emb_t = embedder.embed_single(target.sequence)
    wt_res_t = sample_ensemble_with_trajectory(
        model, wt_emb_t, mut_pos=0, mut_aa=0,
        n_samples=64, n_trajectory_samples=16, device=device,
    )
    wt_rg_t = radius_of_gyration(torch.from_numpy(wt_res_t['ensemble'])).mean().item()

    for mut in target.mutations:
        mut_seq = target.mutant_sequence(mut)
        mut_emb = embedder.embed_single(mut_seq)
        aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut.mt, 0)
        res = sample_ensemble_with_trajectory(
            model, mut_emb, mut_pos=mut.pos, mut_aa=aa_idx,
            n_samples=32, n_trajectory_samples=4, device=device,
        )
        mut_rg = radius_of_gyration(torch.from_numpy(res['ensemble'])).mean().item()
        traj_feats = extract_trajectory_features(res['trajectory'], mutation_pos=mut.pos - 1)
        llr = scorer.score_mutation(target.sequence, mut.pos, mut.wt, mut.mt, fast=True)
        disease_data.append({
            'target': tid, 'mutation_id': mut.id,
            'pos': mut.pos, 'wt': mut.wt, 'mt': mut.mt,
            'agg_rate': mut.agg_rate_relative,
            'delta_rg': mut_rg - wt_rg_t, 'llr_site': llr['llr_site'],
            **{k: v for k, v in traj_feats.items() if not k.startswith('_')},
        })
        print(f'  {mut.id}: done')

from brain_idp_flow.analysis.ml_predictor import run_cross_protein_transfer
all_combined = scored_dms + disease_data
print(f'\nCombined: {len(all_combined)} mutations')
transfer_results = run_cross_protein_transfer(all_combined)

In [ ]:
# CELL 12: IAPP DMS 데이터 로드
import subprocess, pandas as pd

IAPP_WT = "KCNTATCATQRLANFLVHSSNNFGAILSSTNVGSNTY"
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

# RData 다운로드
for fname, url in {
    "MAVE_IAPP_fitness_replicates.RData":
        "https://raw.githubusercontent.com/BEBlab/MAVE-IAPP/main/Required%20files/MAVE_IAPP_fitness_replicates.RData",
    "INDEL_datasets_AB.Rdata":
        "https://raw.githubusercontent.com/BEBlab/MAVE-IAPP/main/Required%20files/INDEL_datasets_AB.Rdata",
}.items():
    if not os.path.exists(fname):
        subprocess.run(["wget", "-q", url, "-O", fname], check=True)

# singles.df export
r_fix = '''
load("MAVE_IAPP_fitness_replicates.RData")
load("INDEL_datasets_AB.Rdata")
write.csv(singles.df, "iapp_dms_fitness.csv", row.names=FALSE)
cat("Exported singles.df:", nrow(singles.df), "rows\\n")
'''
with open("_fix.R", "w") as f:
    f.write(r_fix)
r = subprocess.run(["Rscript", "_fix.R"], capture_output=True, text=True)
print(r.stdout); os.remove("_fix.R")

# CSV 로드 + 파싱
df = pd.read_csv("iapp_dms_fitness.csv", low_memory=False)
print(f"Rows: {len(df)}, Columns: {list(df.columns)}")

iapp_dms = []
for _, row in df.iterrows():
    wt = str(row['WT_AA']).strip()
    mt = str(row['Mut']).strip()
    pos = int(row['Pos'])
    if wt not in VALID_AA or mt not in VALID_AA or wt == mt:
        continue
    if pos < 1 or pos > len(IAPP_WT):
        continue
    ns = row['nscore_c']
    if pd.isna(ns):
        continue
    sigma = float(row['sigma']) if not pd.isna(row['sigma']) else 0.0
    iapp_dms.append({
        'pos': pos, 'wt': wt, 'mt': mt,
        'mutation_id': f"{wt}{pos}{mt}",
        'nucleation_score': float(ns), 'sigma': sigma,
        'agg_rate': float(np.exp(np.clip(float(ns), -5, 5))),
        'is_fad': bool(row.get('fAD', False)),
        'target': 'iapp', 'source': 'Badia2026',
    })
print(f'IAPP DMS loaded: {len(iapp_dms)} substitutions')

In [ ]:
# CELL 13: IAPP 35M + 650M Scoring
# 35M
print("=== IAPP 35M Scoring ===")
wt_emb_iapp = embedder.embed_single(IAPP_WT)
wt_result_iapp = sample_ensemble_with_trajectory(
    model, wt_emb_iapp, mut_pos=0, mut_aa=0,
    n_samples=64, n_trajectory_samples=0, device=device,
)
wt_rg_iapp = radius_of_gyration(torch.from_numpy(wt_result_iapp['ensemble'])).numpy()
wt_rg_mean_iapp = wt_rg_iapp.mean()
print(f"IAPP WT Rg (35M): {wt_rg_mean_iapp:.2f}")

scored_iapp_35m = []
for i, mut in enumerate(iapp_dms):
    mut_seq = list(IAPP_WT)
    mut_seq[mut['pos'] - 1] = mut['mt']
    mut_seq = ''.join(mut_seq)
    mut_emb = embedder.embed_single(mut_seq)
    aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut['mt'], 0)
    result = sample_ensemble_with_trajectory(
        model, mut_emb, mut_pos=mut['pos'], mut_aa=aa_idx,
        n_samples=32, n_trajectory_samples=0, device=device,
    )
    mut_rg = radius_of_gyration(torch.from_numpy(result['ensemble'])).numpy()
    scored_iapp_35m.append({**mut, 'delta_rg': mut_rg.mean() - wt_rg_mean_iapp})
    if (i + 1) % 100 == 0:
        print(f"  35M: {i+1}/{len(iapp_dms)}")
print(f"35M Done: {len(scored_iapp_35m)}")

# 650M
print("\n=== IAPP 650M Scoring ===")
wt_emb_iapp_650 = embedder_650m.embed_single(IAPP_WT)
wt_result_iapp_650 = sample_ensemble_with_trajectory(
    model_650m, wt_emb_iapp_650, mut_pos=0, mut_aa=0,
    n_samples=64, n_trajectory_samples=0, device=device,
)
wt_rg_iapp_650 = radius_of_gyration(torch.from_numpy(wt_result_iapp_650['ensemble'])).numpy()
wt_rg_mean_iapp_650 = wt_rg_iapp_650.mean()
print(f"IAPP WT Rg (650M): {wt_rg_mean_iapp_650:.2f}")

scored_iapp_650m = []
for i, mut in enumerate(iapp_dms):
    mut_seq = list(IAPP_WT)
    mut_seq[mut['pos'] - 1] = mut['mt']
    mut_seq = ''.join(mut_seq)
    mut_emb = embedder_650m.embed_single(mut_seq)
    aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut['mt'], 0)
    result = sample_ensemble_with_trajectory(
        model_650m, mut_emb, mut_pos=mut['pos'], mut_aa=aa_idx,
        n_samples=32, n_trajectory_samples=0, device=device,
    )
    mut_rg = radius_of_gyration(torch.from_numpy(result['ensemble'])).numpy()
    scored_iapp_650m.append({**mut, 'delta_rg_650m': mut_rg.mean() - wt_rg_mean_iapp_650})
    if (i + 1) % 100 == 0:
        print(f"  650M: {i+1}/{len(iapp_dms)}")
print(f"650M Done: {len(scored_iapp_650m)}")

In [ ]:
# CELL 14: Multi-Amyloid Comparison (Abeta42 vs IAPP)
ns_ab = np.array([d['nucleation_score'] for d in scored_dms])
ns_iapp = np.array([d['nucleation_score'] for d in scored_iapp_35m])
drg_ab = np.array([d['delta_rg'] for d in scored_dms])
drg_iapp = np.array([d['delta_rg'] for d in scored_iapp_35m])
drg_iapp_650 = np.array([d['delta_rg_650m'] for d in scored_iapp_650m])
drg_ab_650 = np.array([d['delta_rg_650m'] for d in scored_650m])

print("=" * 60)
print("SUMMARY: MULTI-AMYLOID delta_Rg CORRELATION")
print("=" * 60)
print(f"{'Protein':<12} {'Model':<8} {'n':>6} {'rho':>8} {'p-value':>12}")
print("-" * 60)
for name, drg, ns_arr in [
    ('Abeta42', drg_ab, ns_ab), ('Abeta42 650M', drg_ab_650, ns_ab),
    ('IAPP', drg_iapp, ns_iapp), ('IAPP 650M', drg_iapp_650, ns_iapp),
]:
    model_name = '650M' if '650M' in name else '35M'
    prot = name.replace(' 650M', '')
    rho, p = spearmanr(drg, ns_arr)
    print(f"  {prot:<12} {model_name:<8} {len(ns_arr):>6} {rho:>8.3f} {p:>12.2e}")
print("=" * 60)

# 플롯
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for col, (drg, ns_arr, title, n, c) in enumerate([
    (drg_ab, ns_ab, 'Abeta42 (35M)', len(ns_ab), 'steelblue'),
    (drg_iapp, ns_iapp, 'IAPP (35M)', len(ns_iapp), 'steelblue'),
]):
    ax = axes[0, col]; rho, p = spearmanr(drg, ns_arr)
    ax.scatter(drg, ns_arr, s=10, alpha=0.4, c=c)
    ax.set_xlabel('delta Rg'); ax.set_ylabel('Nucleation Score')
    ax.set_title(f'{title} (n={n})\nrho={rho:.3f}, p={p:.2e}')
for col, (drg, ns_arr, title, n, c) in enumerate([
    (drg_ab_650, ns_ab, 'Abeta42 (650M)', len(ns_ab), 'coral'),
    (drg_iapp_650, ns_iapp, 'IAPP (650M)', len(ns_iapp), 'coral'),
]):
    ax = axes[1, col]; rho, p = spearmanr(drg, ns_arr)
    ax.scatter(drg, ns_arr, s=10, alpha=0.4, c=c)
    ax.set_xlabel('delta Rg'); ax.set_ylabel('Nucleation Score')
    ax.set_title(f'{title} (n={n})\nrho={rho:.3f}, p={p:.2e}')
fig.suptitle('Multi-Amyloid Validation: Abeta42 vs IAPP', fontsize=16)
fig.tight_layout(); fig.savefig('results/dms/abeta42_vs_iapp.png', dpi=150); plt.show()

# Direction Asymmetry
print("\nDIRECTION ASYMMETRY ANALYSIS")
print("=" * 60)
for name, drg, ns_arr in [('Abeta42', drg_ab, ns_ab), ('IAPP', drg_iapp, ns_iapp)]:
    mask_acc = ns_arr > 0; mask_slow = ns_arr < 0
    if mask_acc.sum() > 10:
        rho, p = spearmanr(drg[mask_acc], ns_arr[mask_acc])
        print(f"{name} accelerating (n={mask_acc.sum()}): rho={rho:.3f}, p={p:.2e}")
    if mask_slow.sum() > 10:
        rho, p = spearmanr(drg[mask_slow], ns_arr[mask_slow])
        print(f"{name} slowing     (n={mask_slow.sum()}): rho={rho:.3f}, p={p:.2e}")

In [ ]:
# CELL 15: CANYA Baseline
!git clone https://github.com/lehner-lab/canya.git /content/canya_repo 2>/dev/null || true
sys.path.insert(0, '/content/canya_repo')
from canya.modeling import get_predictions

# Abeta42 sequences
seqs_ab = []; ids_ab = []
for d in scored_dms:
    seq = list(ABETA_SEQ); seq[d['pos']-1] = d['mt']
    seqs_ab.append(''.join(seq)); ids_ab.append(d['mutation_id'])
seqs_ab.insert(0, ABETA_SEQ); ids_ab.insert(0, 'WT')

# IAPP sequences
seqs_iapp = []; ids_iapp = []
for d in iapp_dms:
    seq = list(IAPP_WT); seq[d['pos']-1] = d['mt']
    seqs_iapp.append(''.join(seq)); ids_iapp.append(d['mutation_id'])
seqs_iapp.insert(0, IAPP_WT); ids_iapp.insert(0, 'WT')

print("Running CANYA on Abeta42...")
preds_ab = get_predictions(seqs_ab, summarize='mean', mode='ensemble')
print("Running CANYA on IAPP...")
preds_iapp = get_predictions(seqs_iapp, summarize='mean', mode='ensemble')
print("CANYA done")

# 결과 매칭
import pandas as pd

# preds가 어떤 형태인지 확인
print(f"preds_ab type: {type(preds_ab)}, shape/len: {preds_ab.shape if hasattr(preds_ab, 'shape') else len(preds_ab)}")

In [ ]:
# CELL 16: CANYA 결과 분석 (CELL 15 출력 보고 조정 필요)
# preds_ab가 array면:
if hasattr(preds_ab, 'shape') or isinstance(preds_ab, np.ndarray):
    canya_ab_scores = np.array(preds_ab).flatten()
    wt_canya_ab = canya_ab_scores[0]
    delta_canya_ab = canya_ab_scores[1:] - wt_canya_ab  # WT 제외

    canya_iapp_scores = np.array(preds_iapp).flatten()
    wt_canya_iapp = canya_iapp_scores[0]
    delta_canya_iapp = canya_iapp_scores[1:] - wt_canya_iapp
# preds_ab가 DataFrame이면:
elif isinstance(preds_ab, pd.DataFrame):
    canya_ab_scores = preds_ab.iloc[:, -1].values
    wt_canya_ab = canya_ab_scores[0]
    delta_canya_ab = canya_ab_scores[1:] - wt_canya_ab

    canya_iapp_scores = preds_iapp.iloc[:, -1].values
    wt_canya_iapp = canya_iapp_scores[0]
    delta_canya_iapp = canya_iapp_scores[1:] - wt_canya_iapp

print("=" * 70)
print("HEAD-TO-HEAD COMPARISON")
print("=" * 70)
print(f"{'Method':<35} {'Protein':<10} {'n':>5} {'rho':>8} {'p-value':>12}")
print("-" * 70)

# Abeta42
rho_c, p_c = spearmanr(delta_canya_ab, ns_ab)
rho_f, p_f = spearmanr(drg_ab, ns_ab)
rho_l, p_l = spearmanr([d.get('llr_site',0) for d in scored_dms], ns_ab)
print(f"  {'CANYA (ensemble mean)':<35} {'Abeta42':<10} {len(ns_ab):>5} {rho_c:>8.3f} {p_c:>12.2e}")
print(f"  {'ESM-2 LLR':<35} {'Abeta42':<10} {len(ns_ab):>5} {rho_l:>8.3f} {p_l:>12.2e}")
print(f"  {'Flow delta_Rg (35M)':<35} {'Abeta42':<10} {len(ns_ab):>5} {rho_f:>8.3f} {p_f:>12.2e}")
print(f"  {'All-7 features (GB CV)':<35} {'Abeta42':<10} {'751':>5} {'0.321':>8} {'---':>12}")

# IAPP
rho_ci, p_ci = spearmanr(delta_canya_iapp, ns_iapp)
rho_fi, p_fi = spearmanr(drg_iapp, ns_iapp)
print(f"  {'CANYA (ensemble mean)':<35} {'IAPP':<10} {len(ns_iapp):>5} {rho_ci:>8.3f} {p_ci:>12.2e}")
print(f"  {'Flow delta_Rg (35M)':<35} {'IAPP':<10} {len(ns_iapp):>5} {rho_fi:>8.3f} {p_fi:>12.2e}")
print("=" * 70)

# 플롯
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for col, (pred, ns_arr, name, color) in enumerate([
    (delta_canya_ab, ns_ab, 'CANYA | Abeta42', 'coral'),
    (drg_ab, ns_ab, 'Flow delta_Rg | Abeta42', 'steelblue'),
]):
    ax = axes[0, col]; rho, p = spearmanr(pred, ns_arr)
    ax.scatter(pred, ns_arr, s=10, alpha=0.4, c=color)
    ax.set_xlabel(name.split('|')[0]); ax.set_ylabel('Nucleation Score')
    ax.set_title(f'{name}\nrho={rho:.3f}, p={p:.2e}')
for col, (pred, ns_arr, name, color) in enumerate([
    (delta_canya_iapp, ns_iapp, 'CANYA | IAPP', 'coral'),
    (drg_iapp, ns_iapp, 'Flow delta_Rg | IAPP', 'steelblue'),
]):
    ax = axes[1, col]; rho, p = spearmanr(pred, ns_arr)
    ax.scatter(pred, ns_arr, s=10, alpha=0.4, c=color)
    ax.set_xlabel(name.split('|')[0]); ax.set_ylabel('Nucleation Score')
    ax.set_title(f'{name}\nrho={rho:.3f}, p={p:.2e}')
fig.suptitle('CANYA vs Flow Model', fontsize=16)
fig.tight_layout(); fig.savefig('results/dms/canya_vs_flow.png', dpi=150); plt.show()

In [ ]:
# CELL 17: Per-Residue Rg Contribution (35M vs 650M)
def per_residue_rg_contribution(coords):
    com = coords.mean(axis=-2, keepdims=True)
    dists_sq = ((coords - com) ** 2).sum(axis=-1)
    rg_sq = dists_sq.mean(axis=-1, keepdims=True)
    return (dists_sq / (rg_sq + 1e-8)).mean(axis=0)

fad_muts = [d for d in scored_dms if d.get('is_fad', False)]
test_muts = fad_muts[:3] if len(fad_muts) >= 3 else fad_muts

fig, axes = plt.subplots(len(test_muts), 2, figsize=(16, 4*len(test_muts)))
if len(test_muts) == 1: axes = axes.reshape(1, -1)

for row, mut in enumerate(test_muts):
    mut_seq = list(ABETA_SEQ); mut_seq[mut['pos']-1] = mut['mt']; mut_seq = ''.join(mut_seq)
    aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut['mt'], 0)

    # 35M
    res_35 = sample_ensemble_with_trajectory(model, embedder.embed_single(mut_seq),
        mut_pos=mut['pos'], mut_aa=aa_idx, n_samples=32, n_trajectory_samples=0, device=device)
    wt_35 = sample_ensemble_with_trajectory(model, wt_emb,
        mut_pos=mut['pos'], mut_aa=aa_idx, n_samples=32, n_trajectory_samples=0, device=device)
    L = min(res_35['ensemble'].shape[1], wt_35['ensemble'].shape[1])
    dc35 = per_residue_rg_contribution(res_35['ensemble'][:,:L,:]) - per_residue_rg_contribution(wt_35['ensemble'][:,:L,:])

    # 650M
    res_650 = sample_ensemble_with_trajectory(model_650m, embedder_650m.embed_single(mut_seq),
        mut_pos=mut['pos'], mut_aa=aa_idx, n_samples=32, n_trajectory_samples=0, device=device)
    wt_650 = sample_ensemble_with_trajectory(model_650m, embedder_650m.embed_single(ABETA_SEQ),
        mut_pos=mut['pos'], mut_aa=aa_idx, n_samples=32, n_trajectory_samples=0, device=device)
    L2 = min(res_650['ensemble'].shape[1], wt_650['ensemble'].shape[1])
    dc650 = per_residue_rg_contribution(res_650['ensemble'][:,:L2,:]) - per_residue_rg_contribution(wt_650['ensemble'][:,:L2,:])

    residues = np.arange(1, L+1)
    for col, (dc, label) in enumerate([(dc35,'35M'),(dc650,'650M')]):
        ax = axes[row, col]
        r = np.arange(1, len(dc)+1)
        colors = ['#e74c3c' if d>0 else '#3498db' for d in dc]
        ax.bar(r, dc, color=colors, width=0.8, alpha=0.7)
        ax.axvline(x=mut['pos'], color='black', linestyle='--', linewidth=2,
                   label=f"mut: {mut['wt']}{mut['pos']}{mut['mt']}")
        ax.axhline(y=0, color='gray', linewidth=0.5)
        ax.set_xlabel('Residue'); ax.set_ylabel('delta Rg Contribution')
        ax.set_title(f"{label} | {mut['wt']}{mut['pos']}{mut['mt']}"); ax.legend(fontsize=9)

fig.suptitle('Per-Residue delta Rg Contribution: 35M vs 650M', fontsize=14)
fig.tight_layout(); fig.savefig('results/dms/per_residue_rg_35m_vs_650m.png', dpi=150); plt.show()

In [ ]:
# CELL 18: 전체 결과 저장 + Drive 백업
import json

final = {
    'abeta42_35m_rho': float(spearmanr(drg_ab, ns_ab)[0]),
    'abeta42_650m_rho': float(spearmanr(drg_ab_650, ns_ab)[0]),
    'iapp_35m_rho': float(spearmanr(drg_iapp, ns_iapp)[0]),
    'iapp_650m_rho': float(spearmanr(drg_iapp_650, ns_iapp)[0]),
    'ml_lasso_cv_rho': dms_ml_results['lasso']['cv_mean_rho'],
    'ml_rf_cv_rho': dms_ml_results['rf']['cv_mean_rho'],
}
with open('results/dms/full_pipeline_results.json', 'w') as f:
    json.dump(final, f, indent=2, default=float)

try:
    import shutil
    backup = '/content/drive/MyDrive/brain_idp_flow_results'
    shutil.copytree('results', backup, dirs_exist_ok=True)
    print(f'Backed up to {backup}')
except:
    print('Drive backup skipped')

print('\n=== FULL PIPELINE COMPLETE ===')